In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# import pandas as pd
# data = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
# data.to_csv("submission.csv", index=False)

**Import modules**

In [ ]:
import subprocess, sys

def pip_install(pkg):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        check=False
    )

pip_install("google-genai")   

import os, re, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd

try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm import tqdm

In [ ]:
GEMINI_MODEL  = "gemini-2.0-flash"   
MAX_WORKERS   = 4                    
MAX_RETRIES   = 5
RETRY_SLEEP   = 5                    
VALID         = set("ABCDE")
FALLBACK      = list("ABCDE")

In [ ]:
ON_KAGGLE = Path("/kaggle/input").exists()

def find_csv(name):
    if ON_KAGGLE:
        hits = list(Path("/kaggle/input").rglob(name))
        if hits:
            return hits[0]
    p = Path(name)
    if p.exists():
        return p
    raise FileNotFoundError(f"'{name}' not found. Attach the competition dataset.")

TEST_PATH = find_csv("test.csv")
OUT_PATH  = Path("/kaggle/working/submission.csv") if ON_KAGGLE else Path("submission.csv")

test = pd.read_csv(TEST_PATH)
print(f" Loaded {len(test)} test rows from {TEST_PATH}")
print(test.head(2).to_string())

In [ ]:
SYSTEM_INSTRUCTION = (
    "You are an expert at solving multiple-choice questions across all subjects. "
    "When given a question and five options (A-E), you output exactly three "
    "space-separated labels ranked from most likely correct to least likely. "
    "Output ONLY the three labels — no explanation, no punctuation. "
    "Example: B A D"
)

def make_prompt(row):
    return (
        f"Question: {row['prompt']}\n\n"
        f"A) {row['A']}\n"
        f"B) {row['B']}\n"
        f"C) {row['C']}\n"
        f"D) {row['D']}\n"
        f"E) {row['E']}\n\n"
        "Rank the top 3 most likely correct options (best first):"
    )

def parse_labels(text):
    """Extract exactly 3 valid labels; pad with unused labels if needed."""
    found, seen = [], set()
    for tok in re.findall(r"\b([A-Ea-e])\b", text):
        tok = tok.upper()
        if tok in VALID and tok not in seen:
            found.append(tok)
            seen.add(tok)
        if len(found) == 3:
            break
    for lab in FALLBACK:
        if len(found) >= 3:
            break
        if lab not in seen:
            found.append(lab)
            seen.add(lab)
    return found[:3]


In [ ]:
def get_gemini_key():
    """Read key from Kaggle Secrets or environment variable."""
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("GEMINI_API_KEY")
        if key and key.strip():
            return key.strip()
    except Exception:
        pass
    return os.environ.get("GEMINI_API_KEY", "").strip()

In [ ]:
class GeminiBackend:
    def __init__(self, api_key):
        import google.genai as genai
        from google.genai import types
        self._types  = types
        self._client = genai.Client(api_key=api_key)
        self._model  = GEMINI_MODEL
        print(f" Using Gemini backend ({self._model})")

    def predict(self, row):
        from google.genai import types
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                resp = self._client.models.generate_content(
                    model    = self._model,
                    contents = make_prompt(row),
                    config   = types.GenerateContentConfig(
                        system_instruction = SYSTEM_INSTRUCTION,
                        temperature        = 0.0,
                        max_output_tokens  = 20,
                    )
                )
                return parse_labels(resp.text.strip())
            except Exception as e:
                err = str(e)
                # Rate limit
                if "429" in err or "quota" in err.lower() or "rate" in err.lower():
                    wait = RETRY_SLEEP * attempt
                    print(f"\n   Rate limit — sleeping {wait}s")
                    time.sleep(wait)
                else:
                    print(f"\n    Gemini error (attempt {attempt}): {err[:80]}")
                    if attempt == MAX_RETRIES:
                        return FALLBACK[:3]
                    time.sleep(RETRY_SLEEP)
        return FALLBACK[:3]


In [ ]:
class SentenceTransformerBackend:
    MODEL_NAME = "all-MiniLM-L6-v2"

    def __init__(self):
        from sentence_transformers import SentenceTransformer
        print(f" Loading sentence-transformers ({self.MODEL_NAME}) …")
        self._model = SentenceTransformer(self.MODEL_NAME)
        print("  Model ready.")

    def predict(self, row):
        opts   = [str(row[o]) for o in "ABCDE"]
        prompt = str(row["prompt"])
        texts  = [prompt] + opts
        embs   = self._model.encode(texts, normalize_embeddings=True)
        sims   = embs[1:] @ embs[0]                        # cosine similarity

        # Blend with length signal
        lens   = np.array([len(o) for o in opts], dtype=float)
        lens  /= lens.max() + 1e-9

        scores = 0.80 * sims + 0.20 * lens
        ranked = [list("ABCDE")[i] for i in np.argsort(scores)[::-1]]
        return ranked[:3]


In [ ]:
class TFIDFBackend:
    def __init__(self):
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.metrics.pairwise import cosine_similarity
        self._tfidf = TfidfVectorizer
        self._cos   = cosine_similarity
        print("Using TF-IDF fallback backend (no API key, no GPU needed).")

    def predict(self, row):
        opts   = [str(row[o]) for o in "ABCDE"]
        prompt = str(row["prompt"])

        # Clean boilerplate from prompts
        for phrase in [
            "Pick the best possible answer:", "Determine the correct option:",
            "among the listed options.", "from the following choices.", "carefully."
        ]:
            prompt = prompt.replace(phrase, "")
        prompt = prompt.strip()

        texts = [prompt] + opts
        try:
            w = self._tfidf(ngram_range=(1,3), sublinear_tf=True).fit_transform(texts)
            s_word = self._cos(w[0:1], w[1:]).flatten()
        except Exception:
            s_word = np.zeros(5)
        try:
            c = self._tfidf(analyzer="char_wb", ngram_range=(3,5),
                            sublinear_tf=True).fit_transform(texts)
            s_char = self._cos(c[0:1], c[1:]).flatten()
        except Exception:
            s_char = np.zeros(5)

        lens  = np.array([len(o) for o in opts], dtype=float)
        s_len = lens / (lens.max() + 1e-9)
        wc    = np.array([len(o.split()) for o in opts], dtype=float)
        s_wc  = wc / (wc.max() + 1e-9)

        scores = 0.40*s_len + 0.30*s_word + 0.20*s_char + 0.10*s_wc
        ranked = [list("ABCDE")[i] for i in np.argsort(scores)[::-1]]
        return ranked[:3]

In [ ]:
gemini_key = get_gemini_key()

if gemini_key:
    try:
        backend = GeminiBackend(gemini_key)
        BACKEND_NAME = "Gemini"
    except Exception as e:
        print(f" Gemini init failed: {e}. Falling back …")
        gemini_key = None

if not gemini_key:
    try:
        backend = SentenceTransformerBackend()
        BACKEND_NAME = "sentence-transformers"
    except Exception as e:
        print(f"  sentence-transformers not available: {e}. Using TF-IDF.")
        backend = TFIDFBackend()
        BACKEND_NAME = "TF-IDF"

print(f"\n→ Active backend: {BACKEND_NAME}\n")

In [ ]:
rows_list = list(test.iterrows())
results   = {}

print(f"Predicting {len(test)} rows …\n")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_to_id = {
        executor.submit(backend.predict, row): int(row["id"])
        for _, row in rows_list
    }
    with tqdm(total=len(future_to_id), desc=f"[{BACKEND_NAME}]") as pbar:
        for future in as_completed(future_to_id):
            rid = future_to_id[future]
            try:
                preds = future.result()
            except Exception as e:
                print(f"\n  Row {rid} failed: {e}")
                preds = FALLBACK[:3]
            results[rid] = " ".join(preds)
            pbar.update(1)


In [ ]:
submission = pd.DataFrame(
    [{"ID": rid, "Prediction": pred} for rid, pred in sorted(results.items())]
)
submission.to_csv(OUT_PATH, index=False)

print(f"\n  submission.csv saved → {OUT_PATH}")
print(f"    Rows : {len(submission)}")
print(f"    Backend used: {BACKEND_NAME}")
print("\nPreview:")
print(submission.head(10).to_string(index=False))

In [3]:
# train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
# test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")


In [4]:
# print(train_df.shape)
# print(test_df.shape)

(2000, 8)
(500, 7)


In [7]:
# print(train_df.columns.tolist())
# print(test_df.columns.tolist())

['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']
['id', 'prompt', 'A', 'B', 'C', 'D', 'E']
